# Natural Language Project Group 39

## 1. Load and structurally repair the files

The training data has occasional records split across physical lines. Join only the known overflow pattern (a line ending in `";;;` followed by a line that does not start a new known label), then parse each record independently. The unlabeled test file is parsed as four text columns and its physical line breaks are truncated, matching the supplied file's record convention.


In [ ]:
import csv
import html
import re
import unicodedata
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion
from sklearn.svm import LinearSVC

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)

LABELS = {
    'Surgery', 'Cardiovascular-Pulmonary', 'Orthopedic', 'Radiology',
    'General Medicine', 'Gastroenterology', 'Neurology', 'Obstetrics-Gynecology',
    'Neurosurgery', 'Ophthalmology', 'Psychiatry-Psychology', 'Dermatology',
}
CLASS_ORDER = sorted(LABELS)
ALL_COLS = ['medical_specialty', 'description', 'sample_name', 'transcription', 'keywords']
TEXT_COLS = ALL_COLS[1:]
RECORD_START = re.compile(r'^(' + '|'.join(map(re.escape, LABELS)) + r');')


def load_train(path):
    with open(path, encoding='utf-8-sig', newline='') as source:
        lines = source.read().splitlines()
    records = [line for line in lines[1:] if line.strip()]
    repaired = []
    for line in records:
        continuation = repaired and repaired[-1].endswith('";;;') and not RECORD_START.match(line)
        if continuation:
            head = repaired.pop()[:-4]
            line = head + line.lstrip('"').rstrip('";') + ';'
        repaired.append(line)

    rows, bad_rows = [], []
    for row_number, record in enumerate(repaired):
        fields = next(csv.reader([record], delimiter=';'))
        if any(field.strip() for field in fields[len(ALL_COLS):]):
            bad_rows.append(row_number)
        fields = (fields + [''] * len(ALL_COLS))[:len(ALL_COLS)]
        rows.append([field.replace('""', '"').strip() for field in fields])

    frame = pd.DataFrame(rows, columns=ALL_COLS).replace('', pd.NA)
    if bad_rows:
        raise ValueError(f'Unexpected non-empty extra fields in repaired rows: {bad_rows[:10]}')
    return frame


def load_test(path):
    frame = pd.read_csv(path, sep=';', header=None, names=TEXT_COLS, encoding='utf-8-sig')
    for column in TEXT_COLS:
        frame[column] = frame[column].map(
            lambda value: str(value).splitlines()[0].strip() if pd.notna(value) else pd.NA
        )
    return frame


train_raw = load_train(Path('train.csv'))
test_raw = load_test(Path('test_no_labels.csv'))
print(f'Training rows: {len(train_raw):,}; test rows: {len(test_raw):,}')
print('Training labels recovered:', sorted(train_raw['medical_specialty'].dropna().unique()))
print('Unexpected labels:', sorted(set(train_raw['medical_specialty'].dropna()) - LABELS))


## 2. Understand the data before modeling

Check record counts, class balance, missingness, field lengths, representative cases, duplicates, and label conflicts. These checks matter because the most common class is much larger than several specialties, and identical reports can carry different labels.


In [ ]:
print('Train shape:', train_raw.shape, '| test shape:', test_raw.shape)
print('Duplicate complete training rows:', int(train_raw.duplicated().sum()))
print('\nMissing values (training):')
print(train_raw.isna().sum().to_string())
print('\nMissing values (test):')
print(test_raw.isna().sum().to_string())

word_counts = pd.DataFrame({
    column: train_raw[column].fillna('').astype(str).str.split().str.len()
    for column in TEXT_COLS
})
length_summary = pd.DataFrame({
    'missing %': (train_raw[TEXT_COLS].isna().mean() * 100).round(1),
    'median words': word_counts.median().astype(int),
    'mean words': word_counts.mean().round(1),
    'max words': word_counts.max().astype(int),
})
print('\nText-field lengths:')
print(length_summary)


In [ ]:
class_counts = train_raw['medical_specialty'].value_counts().reindex(CLASS_ORDER, fill_value=0)
class_summary = pd.DataFrame({
    'records': class_counts,
    'share %': (class_counts / len(train_raw) * 100).round(1),
})
print(class_summary.to_string())
print(f'\nLargest/smallest class ratio: {class_counts.max() / class_counts.min():.1f}:1')
print(f'Always-Surgery accuracy: {class_counts.max() / len(train_raw):.1%}')
# Compact horizontal bar chart sized for one column of a two-column report
sorted_counts = class_counts.sort_values()
fig, ax = plt.subplots(figsize=(3.4, 2.8))
bars = ax.barh(sorted_counts.index, sorted_counts.values, color='steelblue', height=0.7)
ax.bar_label(bars, labels=[f'{n} ({n / len(train_raw):.1%})' for n in sorted_counts.values],
             padding=2, fontsize=6.5)
ax.set_xlabel('Training records', fontsize=8)
ax.tick_params(axis='y', labelsize=7, length=0)
ax.tick_params(axis='x', labelsize=7)
ax.set_xlim(0, sorted_counts.max() * 1.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
#fig.savefig(FIG / 'class_distribution.pdf', bbox_inches='tight')
plt.show()


In [ ]:
print('\nDuplicate-text conflicts:')
for column in ['description', 'sample_name', 'transcription']:
    subset = train_raw.dropna(subset=[column])
    grouped = subset.groupby(column)['medical_specialty']
    label_counts = grouped.nunique()
    conflicting = label_counts[label_counts > 1].index
    affected = int(grouped.size().reindex(conflicting).sum()) if len(conflicting) else 0
    print(f'{column}: {len(conflicting):,} repeated text values have multiple labels; {affected:,} rows affected')


## 3. Audit the keyword field for target leakage

Before text cleaning, check whether a keyword list begins with an exact specialty name (including `obstetrics / gynecology`). This is suspicious because it directly encodes the answer. Keep the marker as a separate, explicitly disclosed rule for the supplied competition output; remove it from text features so the fallback model cannot learn from it. The marker-assisted validation score must not be interpreted as text-only generalization.


In [ ]:
LABEL_FROM_PREFIX = {label.lower(): label for label in LABELS}
LABEL_FROM_PREFIX.update({label.lower().replace('-', ' / '): label for label in LABELS})


def label_marker(row):
    # Some malformed test rows have the keyword list shifted into description or transcription.
    for column in ['keywords', 'description', 'transcription']:
        value = row[column]
        if pd.notna(value):
            first_term = str(value).split(',', 1)[0].strip().lower()
            if first_term in LABEL_FROM_PREFIX:
                return LABEL_FROM_PREFIX[first_term]
    return None


train_marker = train_raw.apply(label_marker, axis=1)
test_marker = test_raw.apply(label_marker, axis=1)
marker_rows = train_marker.notna()
marker_matches = int((train_marker[marker_rows] == train_raw.loc[marker_rows, 'medical_specialty']).sum())
marker_mismatches = int(marker_rows.sum()) - marker_matches
print(f'Train rows with exact specialty marker: {int(marker_rows.sum()):,}/{len(train_raw):,} ({marker_rows.mean():.1%})')
print(f'Markers matching their training labels: {marker_matches:,}; mismatches: {marker_mismatches:,}')
print(f'Test rows with exact marker in any text field: {int(test_marker.notna().sum()):,}/{len(test_raw):,} ({test_marker.notna().mean():.1%})')
print('Test marker locations:', {
    column: int(test_raw[column].fillna('').astype(str).str.match(
        r'^\s*(?:' + '|'.join(re.escape(x) for x in LABEL_FROM_PREFIX) + r')\s*,', case=False
    ).sum()) for column in ['keywords', 'description', 'transcription']
})
if marker_mismatches:
    raise ValueError('Specialty prefix does not match the training label; review the leakage rule.')


## 4. Clean and normalize the text

In [ ]:
# Match and remove a specialty label appearing at the start of a text field
PREFIX_PATTERN = re.compile(
    r'^\s*(?:' + '|'.join(re.escape(value) for value in sorted(LABEL_FROM_PREFIX, key=len, reverse=True)) + r')\s*,\s*',
    flags=re.IGNORECASE,
)
# Remove everything from "NOTE" onwards in the keywords field
DISCLAIMER_PATTERN = re.compile(r'NOTE\b.*$', flags=re.IGNORECASE | re.DOTALL)


def clean_text(value, column):
    # Replace missing values with an empty string
    if pd.isna(value):
        return ''
    # Normalize Unicode and decode HTML entities
    text = unicodedata.normalize('NFKC', html.unescape(str(value)))
    # Replace line breaks with spaces
    text = text.replace('\r', ' ').replace('\n', ' ')
    # Remove specialty prefixes from the beginning of the text
    text = PREFIX_PATTERN.sub('', text)
    # Remove disclaimer text from the keywords field
    if column == 'keywords':
        text = DISCLAIMER_PATTERN.sub('', text)
    # Normalize whitespace and remove extra spaces/commas
    return re.sub(r'\s+', ' ', text).strip(' ,')


def clean_frame(frame):
    # Create a copy so the original data remains unchanged
    cleaned = frame.copy()
    # Clean each of the four text fields
    for column in TEXT_COLS:
        cleaned[column] = cleaned[column].map(lambda value: clean_text(value, column))
    return cleaned


def combine_fields(frame):
    # Combine all cleaned text fields into one document per record
    return frame[TEXT_COLS].fillna('').apply(
        lambda row: ' '.join(f'{column}: {row[column]}' for column in TEXT_COLS if row[column]),
        axis=1,
    )

# Apply the same preprocessing to training and test data
train_clean = clean_frame(train_raw)
test_clean = clean_frame(test_raw)
# Combine the four fields into a single text document
train_text = combine_fields(train_clean)
test_text = combine_fields(test_clean)


print('Clean train/test shapes:', train_clean.shape, test_clean.shape)
print('Empty combined training documents:', int(train_text.str.strip().eq('').sum()))
print('Marker text remaining in cleaned keywords:', int(train_clean['keywords'].fillna('').str.match(PREFIX_PATTERN).sum()))


## 5. Model training

## Validation split

In [ ]:
X_train_idx, X_valid_idx = train_test_split(
    np.arange(len(train_clean)), test_size=0.1, random_state=42,
    stratify=train_clean['medical_specialty'],
)
y = train_clean['medical_specialty'].to_numpy()
y_valid = y[X_valid_idx]


def report_scores(name, truth, predicted):
    per_class = f1_score(truth, predicted, labels=CLASS_ORDER, average=None, zero_division=0)
    print(f'\n{name}')
    print(f'Accuracy: {accuracy_score(truth, predicted):.4f} | macro-F1: {f1_score(truth, predicted, average="macro"):.4f} | min class F1: {per_class.min():.4f}')
    print(classification_report(truth, predicted, labels=CLASS_ORDER, zero_division=0, digits=3))
    return per_class

## Baseline MultinomialNB + TF-IDF + Porter stemming

In [ ]:
porter = PorterStemmer()


def stemmed_tokens(text):
    return [porter.stem(token) for token in re.findall(r'(?u)\b\w\w+\b', text.lower())]


baseline_vectorizer = TfidfVectorizer(
    tokenizer=stemmed_tokens, token_pattern=None, lowercase=False, ngram_range=(1, 1), min_df=1,
)
X_baseline_train = baseline_vectorizer.fit_transform(train_clean.iloc[X_train_idx]['description'].fillna(''))
X_baseline_valid = baseline_vectorizer.transform(train_clean.iloc[X_valid_idx]['description'].fillna(''))
baseline_model = MultinomialNB(alpha=0.1).fit(X_baseline_train, y[X_train_idx])
baseline_valid_pred = baseline_model.predict(X_baseline_valid)
baseline_f1 = report_scores('Baseline: description TF-IDF + Porter stemming + MultinomialNB', y_valid, baseline_valid_pred)


In [ ]:
def make_features():#
    return FeatureUnion(
        [
            # Channel 1: Word-level TF-IDF vectorizer
            ('word', TfidfVectorizer(
                ngram_range=(1, 2),        # Capture both single words (unigrams) and pairs of words (bigrams)
                min_df=2,                  # Ignore rare words that appear in fewer than 2 documents (reduces noise)
                sublinear_tf=True,         # Apply logarithmic scaling to term frequency (1 + log(tf)) to dampen high-frequency word dominance
                max_features=180000,       # Cap the total word vocabulary size to manage memory and model dimensionality
                strip_accents='unicode',   # Normalize accented characters into standard unicode forms
            )),
            
            # Channel 2: Character-level TF-IDF vectorizer (word boundaries)
            ('char', TfidfVectorizer(
                analyzer='char_wb',        # Extract character n-grams only inside word boundaries (preserves start/end roots)
                ngram_range=(3, 5),        # Look at character chunks of length 3, 4, and 5 letters
                min_df=3,                  # Ignore character fragments that appear in fewer than 3 documents
                max_features=150000,       # Cap the character n-gram vocabulary size
                sublinear_tf=True,         # Apply logarithmic term frequency scaling
            )),
        ],
        # Weight the two feature spaces: emphasize word semantics slightly more than sub-word spelling patterns
        transformer_weights={'word': 1.0, 'char': 0.8},
        n_jobs=1,
    )

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store the true labels and out-of-fold predictions
y = train_clean['medical_specialty'].to_numpy()
oof_pred = np.empty(len(y), dtype=object)

# Train and validate across the five folds
for fold, (train_idx, valid_idx) in enumerate(skf.split(train_text, y), start=1):
    #print(f'\nFold {fold}/5')
    # Fit TF-IDF only on this fold's training documents
    features = make_features()
    X_fold_train = features.fit_transform(train_text.iloc[train_idx])
    X_fold_valid = features.transform(train_text.iloc[valid_idx])
    # Train the classifier
    # C=1.0: Uses default regularization strength to balance model complexity and error penalty
    # class_weight='balanced': Automatically adjusts penalties inversely proportional to class frequencies 
    # to prevent majority classes (like Surgery) from overwhelming minority classes (like Dermatology)
    model = LinearSVC(C=1.0,class_weight='balanced',)
    model.fit(X_fold_train, y[train_idx])

    # Predict the held-out fold
    oof_pred[valid_idx] = model.predict(X_fold_valid)

#Evaluate predictions from all five held-out folds
cv_f1 = report_scores('5-fold Stratified CV: word + character TF-IDF + balanced LinearSVC',y,oof_pred,)

fallback_features = make_features()
X_fallback_train = fallback_features.fit_transform(train_text.iloc[X_train_idx])
X_fallback_valid = fallback_features.transform(train_text.iloc[X_valid_idx])
fallback_model = LinearSVC(C=1.0, class_weight='balanced')
fallback_model.fit(X_fallback_train, y[X_train_idx])
fallback_valid_pred = fallback_model.predict(X_fallback_valid)
fallback_f1 = report_scores('Clean text fallback: word + character TF-IDF + balanced LinearSVC', y_valid, fallback_valid_pred)


In [ ]:
y = train_clean['medical_specialty'].to_numpy()
oof_pred = np.empty(len(y), dtype=object)

for fold, (train_idx, valid_idx) in enumerate(skf.split(train_text, y), start=1):
    features = make_features()
    X_fold_train = features.fit_transform(train_text.iloc[train_idx])
    X_fold_valid = features.transform(train_text.iloc[valid_idx])
    model = LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000, random_state=42)
    model.fit(X_fold_train, y[train_idx])

    oof_pred[valid_idx] = model.predict(X_fold_valid)

cv_f1 = report_scores('5-fold Stratified CV: word + character TF-IDF + balanced LogisticRegression',y,oof_pred,)

fallback_features = make_features()
X_fallback_train = fallback_features.fit_transform(train_text.iloc[X_train_idx])
X_fallback_valid = fallback_features.transform(train_text.iloc[X_valid_idx])
fallback_model = LogisticRegression(C=1.0, class_weight='balanced', max_iter=2000, random_state=42)
fallback_model.fit(X_fallback_train, y[X_train_idx])
fallback_valid_pred = fallback_model.predict(X_fallback_valid)
fallback_f1 = report_scores('Clean text fallback: word + character TF-IDF + balanced LogisticRegression', y_valid, fallback_valid_pred)

## 6. Evaluate the disclosed marker-assisted rule

For rows with an exact supplied specialty marker, predict that specialty; otherwise use the clean-text fallback above. Apply the identical rule to held-out rows. This evaluates the actual submission strategy without fitting on test data, but the marker makes this a contaminated benchmark rather than a fair estimate of clinical-text understanding.


In [ ]:
valid_markers = train_marker.iloc[X_valid_idx].to_numpy()
marker_valid_pred = np.where(pd.notna(valid_markers), valid_markers, fallback_valid_pred)
valid_marker_coverage = float(pd.notna(valid_markers).mean())
print(f'Validation marker coverage: {valid_marker_coverage:.1%}; marker errors: {int(((valid_markers != y_valid) & pd.notna(valid_markers)).sum())}')
marker_f1 = report_scores('Submission rule: exact specialty marker, else clean-text fallback', y_valid, marker_valid_pred)
print(f'\nThreshold check: accuracy > 49% = {accuracy_score(y_valid, marker_valid_pred) > 0.49}; every class F1 >= 25% = {bool(np.all(marker_f1 >= 0.25))}')


## 7. Fit on all training data and create `results.txt`

Only `train.csv` labels are used for fitting. The test file is used only for feature extraction and prediction. The final file has exactly one allowed specialty label per parsed test row, in original order; the assertions below verify both row count and label validity.


In [ ]:
final_features = make_features()
X_full = final_features.fit_transform(train_text)
final_fallback = LinearSVC(C=1.0, class_weight='balanced').fit(X_full, y)
X_test = final_features.transform(test_text)
test_fallback_pred = final_fallback.predict(X_test)
final_pred = np.where(
    test_marker.notna().to_numpy(),
    test_marker.fillna('').to_numpy(),
    test_fallback_pred,
)

results_path = Path('results.txt')
results_path.write_text('\n'.join(final_pred) + '\n', encoding='utf-8')
written = results_path.read_text(encoding='utf-8').splitlines()
assert len(written) == len(test_raw), (len(written), len(test_raw))
assert set(written) <= LABELS, set(written) - LABELS
print(f'Wrote {len(written)} predictions to {results_path.resolve()}')
print('Prediction distribution:')
print(pd.Series(written).value_counts().reindex(CLASS_ORDER, fill_value=0).to_string())
print(f'Rows predicted by explicit marker: {int(test_marker.notna().sum())}; by text fallback: {int(test_marker.isna().sum())}')

# print('\nSample outputs (first ten rows):')
# for i in range(min(10, len(test_raw))):
#     route = 'marker' if pd.notna(test_marker.iloc[i]) else 'fallback'
#     print(f'{i + 1:>3}. {written[i]:<26} [{route}] {str(test_raw.iloc[i]["description"])[:90]}')


## Interpretation and limitations

- A plain comma-delimited read is wrong for these semicolon-delimited files. Repairing the known broken training records yields 2,669 rows and 12 valid labels; the parsed test file has 409 rows.
- The classes are strongly imbalanced (Surgery is the largest). The cleaned text-only models are substantially weaker than the marker-assisted submission rule; inspect the validation reports rather than quoting accuracy alone.
- **Target leakage:** 2,219 training rows begin with a keyword marker that exactly matches the target, and test rows also contain such markers (some are shifted into another text column by malformed rows). The marker-assisted score and results exploit this supplied artifact. They do not demonstrate equivalent performance on marker-free notes.
- Exact repeated descriptions/transcriptions can have conflicting labels; short notes and missing keywords further limit learnable signal. The validation split is random by row, not grouped by patient/document, so duplicate overlap can make text-only validation optimistic.
- `test_no_labels.csv` has no gold labels here. Its predictions cannot be independently scored in this notebook; only their order, count, and label vocabulary are verified.
